# 14 · Views & Indexes

- **Views** are saved queries you can treat like a table — great for reuse.
- **Indexes** speed up lookups and joins on large tables.

> Views/indexes here are prefixed `demo_` and dropped at the end.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## Creating a view
A view named `demo_order_values` that computes each order's total. Query it like a table afterwards.

In [ ]:
%%sql
DROP VIEW IF EXISTS demo_order_values;
CREATE VIEW demo_order_values AS
SELECT o.order_id, o.customer_id, o.order_date,
       SUM(oi.quantity * oi.unit_price) AS order_total
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.customer_id, o.order_date;

SELECT * FROM demo_order_values ORDER BY order_total DESC LIMIT 5;

## Views compose
Build on the view like any table — e.g. total revenue per customer via the view:

In [ ]:
%%sql
SELECT cu.first_name, cu.last_name, ROUND(SUM(v.order_total), 2) AS revenue
FROM demo_order_values v
JOIN customers cu ON v.customer_id = cu.customer_id
GROUP BY v.customer_id
ORDER BY revenue DESC
LIMIT 5;

## Indexes
An index is a lookup structure the database uses to find rows fast. On big
tables, an index on frequently-filtered/joined columns can turn a full scan into
an instant lookup. Create one on `orders.customer_id`:

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_orders_customer;
CREATE INDEX demo_idx_orders_customer ON orders(customer_id);
SELECT 'index created' AS status;

## Did the planner use it?
`EXPLAIN QUERY PLAN` shows how SQLite will run a query. After creating the index, a lookup by customer_id can use it (`SEARCH ... USING INDEX`).

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM orders WHERE customer_id = 1;

### When to index
Index columns you frequently filter or join on. Indexes cost storage and slow
down writes slightly, so don't index everything — index what your queries
actually use.

## Clean up

In [ ]:
%%sql
DROP VIEW IF EXISTS demo_order_values;
DROP INDEX IF EXISTS demo_idx_orders_customer;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Create a view demo_expensive_products listing products with unit_price > 75 (name, price). Then select from it ordered by price desc.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP VIEW IF EXISTS demo_expensive_products;
CREATE VIEW demo_expensive_products AS
SELECT product_name, unit_price FROM products WHERE unit_price > 75;
SELECT * FROM demo_expensive_products ORDER BY unit_price DESC;

### ✅ Recap
Views save and reuse queries as virtual tables; indexes accelerate reads on the
columns you filter and join by. Inspect plans with `EXPLAIN QUERY PLAN`.

**Next:** `15_transactions.ipynb`.